# JavaScript — Classes

## LESSON 33 — Classes

You already know how to write one object by hand. A **class** is a recipe for making many objects that share the same shape and the same methods. You write the recipe once, then use it as many times as you need.

```js
class Product {
  constructor(name, price) {
    this.name = name;
    this.price = price;
  }

  describe() {
    return `${this.name} costs ${this.price}`;
  }
}

const laptop = new Product("Laptop", 1200);

laptop.name;          // "Laptop"
laptop.describe();    // "Laptop costs 1200"
```

- `constructor` runs once, when you write `new`. Its job is to set up the object.
- `this` is the object being built or used — `laptop`, in the lines above.
- Methods are written without `function` and without commas between them.

### static

A `static` member belongs to the class itself, not to the objects it makes. Use it for helpers that do not need one particular object.

```js
class Temperature {
  static fromFahrenheit(f) {
    return new Temperature((f - 32) / 1.8);
  }

  constructor(celsius) {
    this.celsius = celsius;
  }
}

Temperature.fromFahrenheit(212).celsius;   // 100
```

### Getters

A **getter** is a method you read like a property. It looks like a value from the outside, but it runs like a function.

```js
class Cart {
  constructor(items) {
    this.items = items;
  }

  get total() {
    return this.items.reduce((sum, item) => sum + item.price, 0);
  }
}

cart.total;     // no parentheses — it is read like a property
```

### Private fields and methods

Everything written so far is public. Anything holding the object can read `laptop.price`, or overwrite it with nonsense.

A name that starts with `#` is **private**: it exists only inside the class body.

```js
class Account {
  #balance = 0;

  constructor(startingBalance) {
    this.#balance = startingBalance;
  }

  deposit(amount) {
    if (!this.#isValid(amount)) return this.#balance;
    this.#balance += amount;
    return this.#balance;
  }

  #isValid(amount) {
    return typeof amount === "number" && amount > 0;
  }

  get balance() {
    return this.#balance;
  }
}

const account = new Account(100);

account.deposit(50);    // 150
account.balance;        // 150   -> readable, through the getter
account.deposit(-5);    // 150   -> the private check refused it
```

There are two kinds of private member:

- `#balance` is a private **field** — a value only the class body can read.
- `#isValid` is a private **method** — a helper nobody outside the class needs to know about.

So the class decides what the outside world may do:

- Reading works, but only through the `balance` getter.
- Writing does not work at all. There is no setter, so `account.balance = 0` throws a `TypeError` instead of replacing the value.

This is the same idea as the factory function in LESSON 28, where the value lived in a local variable nothing outside could name. A `#` field does the same job: the local variable becomes a declared field, and the closure becomes the class body. The difference is that `#` needs less ceremony, and it works for every object the class makes.

Four rules are worth memorizing.

**1. Every `#` name must be declared in the class body.** `this.name = name` may invent a public key on the fly, but a private field cannot be invented that way. `this.#total = 0` with no `#total;` line in the class is a `SyntaxError: Private field '#total' must be declared in an enclosing class`.

**2. From outside, `#` is not `undefined` — it is a `SyntaxError`.** Reading a missing key gives you `undefined`. Reading a private field from outside does not: `account.#balance` refuses to run at all.

> One caveat for this notebook. The Deno kernel transpiles each cell before running it, and that lets the line slip through. Everywhere real JavaScript runs — a browser, a `.js` file, a server — it is a hard error. Trust the rule, not the notebook on this one point.

**3. Private fields are invisible from the outside.** They are not keys, so none of the tools from LESSON 28 can see them:

```js
Object.keys(account);      // []     -> #balance is not a key
const copy = { ...account };   // {}  -> there is nothing to copy
JSON.stringify(account);   // "{}"   -> LESSON 38 covers JSON
```

That last line bites in real projects. An object saved to JSON silently loses everything private. Anything that has to survive being saved must be exposed through a getter or a method.

**4. `#name in obj` asks whether an object really came from this class.** Written inside the class, it is the safe way to check from outside:

```js
class Account {
  #balance = 0;

  static isAccount(value) {
    return #balance in value;
  }
}

Account.isAccount(account);            // true
Account.isAccount({ balance: 100 });   // false   -> a lookalike, not an Account
```

A `static` member can be private too. `static #count = 0;` is the usual way to keep a tally that belongs to the class and to nobody else. Read it through the class name, `Account.#count`, never through `this`.

### Key notes

- **Forgetting `new` breaks everything.** `Product("Laptop", 1200)` without `new` throws in a class. That error message is telling you exactly what is missing.
- **`this` depends on how a method is called, not where it is written.** Pass `laptop.describe` somewhere as a callback and `this` is lost. Pass `() => laptop.describe()` instead, so the call still happens on the object.
- Class names are written in `PascalCase` by convention. It is only a convention, but every JavaScript reader relies on it.
- A getter must **return** something. A getter that only prints gives you `undefined` when you read it.
- **A `#` name must be declared in the class body**, and it only works inside that body. From outside, `obj.#field` is a `SyntaxError`, not `undefined`.
- **Private fields are invisible to `Object.keys`, spread and `JSON.stringify`.** Anything that has to survive being saved must come out through a getter or a method.
- Use `#field in value` inside the class to test whether an object really came from it.

### Example

In [ ]:
class ExampleBook {
  static genres = ["fiction", "essay"];
  static #created = 0;

  #timesRead = 0;

  constructor(title, author, pages) {
    this.title = title;
    this.author = author;
    this.pages = pages;
    ExampleBook.#created += 1;
  }

  describe() {
    return `${this.title} by ${this.author}`;
  }

  read() {
    if (this.#canBeRead()) {
      this.#timesRead += 1;
    }
    return this.#timesRead;
  }

  #canBeRead() {
    return this.pages > 0;
  }

  get isLong() {
    return this.pages > 300;
  }

  get timesRead() {
    return this.#timesRead;
  }

  static get created() {
    return ExampleBook.#created;
  }

  static isBook(value) {
    return #timesRead in value;
  }
}

const exampleBook = new ExampleBook("1984", "Orwell", 328);

console.log(exampleBook.title);
console.log(exampleBook.describe());
console.log(exampleBook.isLong);
console.log(ExampleBook.genres);

exampleBook.read();
exampleBook.read();
console.log(exampleBook.timesRead);

// the public keys are all there; #timesRead and #created never show up
console.log(Object.keys(exampleBook));
console.log(JSON.stringify(exampleBook));
console.log({ ...exampleBook });

console.log(ExampleBook.created);
console.log(ExampleBook.isBook(exampleBook), ExampleBook.isBook({ title: "1984" }));

// exampleBook.#timesRead;   // in a real .js file this line is a SyntaxError

### Exercise

Write a class `Person` that:

1. Takes `name` and `age` in its constructor and stores both.
2. Has a method `greet()` returning `"Hi, I'm <name>"`.
3. Has a getter `isAdult` returning `true` when the age is 18 or over.

Then create two people, one adult and one not, and print the greeting and `isAdult` for each.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here
class Person{
    name;
    age;

    constructor(name,age){
        this.name=name;
        this.age=age;
    }

    greet(){
        return `Hi, I'm ${this.name}`;
    }
    isAdult(){
        return this.age>=18;
    }
}

const p1=new Person("Sam",34);
const p2=new Person("Jimmy",12);

console.log(p1.greet(), p1.isAdult());
console.log(p2.greet(), p2.isAdult());




### Mini challenge

Write a class `Playlist` that:

1. Keeps its songs in a **private** field `#songs`, starting as an empty array.
2. Has `add(title)`, which pushes a song and **returns the playlist itself** so calls can be chained.
3. Has a getter `count`, and a getter `titles` that returns a **copy** of the list rather than the list itself.
4. Has a `static` method `fromArray(titles)` that builds a playlist from an array of titles in one go.

Then build one playlist with chained `add` calls, another with `fromArray`, and print both counts. Finally, push something into what `titles` gave you and print the count again — it should not have changed.

In [ ]:
// Your code here
class Playlist{
    #songs=[];
    constructor(){
    }
    add(title){
        if (typeof(title)==="string" && title.trim().length>0 && !this.#songs.includes(title)){
            this.#songs.push(title);
        }else{
            console.log("Invalid song title or song already exists in the playlist.");
        }
    }
    count(){
        return this.#songs.length;
    }
    titles(){
        return [...this.#songs];
    }
    static fromArray(arr){
        const playlist=new Playlist();
        for(const title of arr){
            playlist.add(title);
        }
        return playlist;        
    }   
}

const myPlaylist=new Playlist();
myPlaylist.add("Song 1");
myPlaylist.add("Song 2");
myPlaylist.add("Song 3");
console.log(myPlaylist.count());
console.log(myPlaylist.titles());

const songArray=["Song A","Song B","Song C"];
const newPlaylist=Playlist.fromArray(songArray);
console.log(newPlaylist.count());
console.log(newPlaylist.titles());

songArray.titles.push("XXX");
console.log(myPlaylist.count());
console.log(myPlaylist.titles());

console.log(Object.keys(myPlaylist)); 


## LESSON 34 — Inheritance and prototypes

When two classes share most of their behavior, one can build on the other instead of repeating it.

```js
class Animal {
  constructor(name) {
    this.name = name;
  }

  speak() {
    return `${this.name} makes a sound`;
  }
}

class Dog extends Animal {
  constructor(name, breed) {
    super(name);          // run the parent constructor FIRST
    this.breed = breed;
  }

  speak() {
    return `${this.name} barks`;    // replaces the parent version
  }
}

const rex = new Dog("Rex", "Beagle");

rex.speak();               // "Rex barks"
rex instanceof Dog;        // true
rex instanceof Animal;     // true   <- still an Animal
```

Three things are doing the work:

- `extends` says "start from that class". `Dog` gets everything `Animal` has.
- `super(...)` calls the parent constructor. In a subclass constructor it must run **before** you touch `this`.
- Writing a method the parent already has **replaces** it, as `speak` does here. To reuse the parent's version inside your own, call `super.speak()`.

### What a prototype actually is

JavaScript has no classes underneath. `class` is friendlier syntax for something older: every object holds a hidden link to another object, its **prototype**.

Reading a property follows that link, in order:

1. JavaScript looks for the property on the object itself.
2. If it is not there, it follows the hidden link to the prototype and looks again.
3. It keeps going up the chain until it finds the property or runs out of links.

```js
rex.speak;                       // not on rex itself
Object.getPrototypeOf(rex);      // Dog.prototype  <- it is found here
```

That chain is why `rex` can use `Animal`'s methods without owning a copy of them. You almost never touch prototypes directly. Knowing the chain exists is still worth it, because it explains two things: why inheritance works at all, and why a method added to a class later is instantly available on objects made earlier.

### Key notes

- **`super()` comes first.** Touching `this` before calling it throws a `ReferenceError`, and the message names the problem plainly.
- Overriding a method **replaces** it. Call `super.method()` inside the override when you want the parent's work as well as your own.
- `instanceof` walks the whole chain, so a `Dog` is also an `Animal`. That is usually what you want.
- Inheritance is easy to overuse. Two classes sharing a *behavior* often want a shared function, not a parent class.

### Example

In [ ]:
class ExampleAccount {
  constructor(owner, balance) {
    this.owner = owner;
    this.balance = balance;
  }

  describe() {
    return `${this.owner}: ${this.balance}`;
  }
}

class ExampleSavings extends ExampleAccount {
  constructor(owner, balance, rate) {
    super(owner, balance);
    this.rate = rate;
  }

  describe() {
    return `${super.describe()} at ${this.rate}%`;
  }
}

const exampleSavings = new ExampleSavings("Mia", 1000, 2.5);

console.log(exampleSavings.describe());
console.log(exampleSavings instanceof ExampleSavings);
console.log(exampleSavings instanceof ExampleAccount);
console.log(Object.getPrototypeOf(exampleSavings) === ExampleSavings.prototype);

### Exercise

Starting from this class:

```js
class Shape {
  constructor(name) {
    this.name = name;
  }

  area() {
    return 0;
  }
}
```

1. Write `Rectangle extends Shape`, taking `width` and `height`, and returning the correct `area()`.
2. Write `Square extends Rectangle`, taking a single `side`.
3. Create one of each and print the name and area of both.
4. Print whether your square is an instance of `Shape`.

_Don't open `solutions.ipynb` until you've actually tried._

In [10]:
// Your code here
class Shape {
  constructor(name) {
    this.name = name;
  }

  area() {
    return 0;
  }
}

class Rectangle extends Shape{
    constructor(width,heigth){
        super("Rectangle");
        this.width=width;
        this.heigth=heigth;
    }
    area(){
        return this.width*this.heigth;
    }
}

class Square extends Rectangle{
    constructor(side){
        super(side,side);
        this.name = "Square";
    }
}

const sh = new Shape("Shape");
const rc = new Rectangle(3,8);
const sq = new Square(4);

console.log(sh.name, sh.area());
console.log(rc.name, rc.area());
console.log(sq.name, sq.area());

console.log(sq instanceof Shape);


Shape 0
Rectangle 24
Square 16
true


### Mini challenge

1. Write a class `Employee` with `name` and `salary`, and a method `report()` returning `"<name> earns <salary>"`.
2. Write `Manager extends Employee`, which also takes a `teamSize`, and whose `report()` reuses the parent's sentence and appends `" and leads <teamSize> people"` — without repeating the first half.
3. Create one of each, put them in an array, loop over it and print each report.

In [17]:
// Your code here
class Employee{
    constructor(name,salary){
        this.name=name;
        this.salary=salary;
    }
    report(){
        return`${this.name} earns ${this.salary}`
    }
}

class Manager extends Employee{
    constructor(name,salary,teamSize){
        super(name,salary);
        this.teamSize=teamSize;
    }
    report(){
        return`${this.name} earns ${this.salary} and leads ${this.teamSize} people`
    }
}

const employee=new Employee("Sam",10000);
const manager=new Manager("Jimmy",1200,33);

const emps= [employee,manager];

for(const emp of emps){
    console.log(emp.report());
}


Sam earns 10000
Jimmy earns 1200 and leads 33 people
